# Wikifonia MIDITok TSD-24 causal LSTM: Top-200 versus Top-250

This notebook trains the same two-layer unidirectional LSTM with Top-200 and Top-250 chord vocabularies. The data split, causal windows, model settings, seed and evaluation protocol are held fixed so that vocabulary size is the principal experimental variable.


## Completion criteria

Run once with `ACTIVE_TOP_N = 200` and once with `ACTIVE_TOP_N = 250`, restarting the runtime between runs. Each run writes its checkpoint, history, metrics, examples, causality results and figures. The final cell verifies comparability and generates the controlled Top-N comparison.


> **Public portfolio copy.** The submitted notebooks had no saved outputs. See `docs/REPRODUCIBILITY.md` for execution steps and `docs/CHANGES_FROM_SUBMISSION.md` for packaging changes. No numerical model results are bundled with this source release.


In [ ]:
%pip -q install "miditok==3.0.6.post1" "symusic==0.6.0" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "pandas>=2.0,<3" "numpy>=1.26,<3" "tqdm>=4.66,<5"

In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import random
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from miditok import TSD
from sklearn.metrics import precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

RANDOM_SEED = 42

def reset_random_state(seed=RANDOM_SEED):
    'Reinitialise every Top-N experiment with the same seed to reduce random variation in the comparison.'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

reset_random_state()


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Random seed:", RANDOM_SEED)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Paths and controlled-comparison settings

Separate Top-200 and Top-250 encoding directories are required. The notebook validates `requested_top_n` in each summary and writes model artifacts to distinct experiment directories.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Set this environment variable before running the notebook to reuse old artifacts.
output_override = os.environ.get("XML_ONLY_OUTPUT_ROOT")
if output_override:
    OUTPUT_ROOT = Path(output_override).expanduser().resolve()
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/melody-to-chord/artifacts")
else:
    OUTPUT_ROOT = PROJECT_ROOT / "artifacts"


ACTIVE_TOP_N = 200
SUPPORTED_TOP_NS = (200, 250)



ENCODING_ROOT_OVERRIDES = {
    200: None,
    250: None,
}

if ACTIVE_TOP_N not in SUPPORTED_TOP_NS:
    raise ValueError(
        f"ACTIVE_TOP_N must be one of {SUPPORTED_TOP_NS}; current value: {ACTIVE_TOP_N}"
    )

EXPECTED_PREPROCESSING_REVISION = "XML_ONLY_V1_2026-08-04"
EXPECTED_PPQ = 24
EXPECTED_SONG_COUNT = 5022
EXPECTED_MIDITOK_VERSION = "3.0.6.post1"
EXPECTED_SYMUSIC_VERSION = "0.6.0"
EXPECTED_MIDITOK_VOCAB_SIZE = 1874
EXPECTED_RETAINED_CHORD_COUNT = ACTIVE_TOP_N
EXPECTED_SUPERVISED_CHORD_CLASSES = ACTIVE_TOP_N + 1  # retained + <UNK>
EXPECTED_CHORD_VOCAB_SIZE = ACTIVE_TOP_N + 2  # <PAD> + <UNK> + retained
COMPATIBILITY_SCHEMA_REVISION = "CAUSAL_TOPN_UNK_AUDIT_V3_2026-08-19"

def find_summary_in_root(root, required_top_n):
    root = Path(root)
    exact = root / "xml_only_miditok_tsd24_topn_unk_v3_summary.json"
    candidates = [exact] if exact.exists() else sorted(root.glob("*summary.json"))
    matches = []
    for path in candidates:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if int(payload.get("requested_top_n", -1)) == int(required_top_n):
            matches.append((path, payload))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(
            f"{root} contains multiple Top-{required_top_n} summaries: "
            f"{[str(path) for path, _ in matches]}"
        )
    return None

def resolve_encoding_root(top_n):
    override = ENCODING_ROOT_OVERRIDES.get(top_n)
    if override is not None:
        root_candidates = [Path(override)]
    else:
        default_candidates = [
            OUTPUT_ROOT / f"encoding_miditok_tsd24_top{top_n}_unk_v3",
            OUTPUT_ROOT / f"encoding_miditok_tsd24_topn_unk_v3_top{top_n}",
            OUTPUT_ROOT / f"encoding_miditok_tsd24_topn{top_n}_unk_v3",
            OUTPUT_ROOT / "encoding_miditok_tsd24_topn_unk_v3",
        ]
        
        discovered = [
            path
            for path in OUTPUT_ROOT.iterdir()
            if path.is_dir()
            and "encoding" in path.name.lower()
            and "unk" in path.name.lower()
            and (
                f"top{top_n}" in path.name.lower()
                or f"top_{top_n}" in path.name.lower()
                or f"topn{top_n}" in path.name.lower()
            )
        ] if OUTPUT_ROOT.exists() else []
        root_candidates = list(dict.fromkeys(default_candidates + sorted(discovered)))

    resolved = []
    for root in root_candidates:
        if not root.exists():
            continue
        result = find_summary_in_root(root, top_n)
        if result is not None:
            summary_path, summary_payload = result
            resolved.append((root, summary_path, summary_payload))

    if not resolved:
        expected = "\n- ".join(str(path) for path in root_candidates)
        raise FileNotFoundError(
            f"Could not find requested_top_n={top_n} encoding. Check:\n- {expected}"
        )

    
    resolved.sort(
        key=lambda item: (
            f"top{top_n}_unk_v3" not in item[0].name,
            str(item[0]),
        )
    )
    return resolved[0]

ENCODING_ROOT, SUMMARY_PATH, encoding_summary_preview = (
    resolve_encoding_root(ACTIVE_TOP_N)
)
SHARD_DIR = ENCODING_ROOT / "shards"

if ENCODING_ROOT.name == "encoding_miditok_tsd24_topn_unk_v3":
    warnings.warn(
        'The generic encoding directory is currently in use. Before switching to another Top-N value, '
        'save the complete directory under a name containing Top200 or Top250; '
        'otherwise, the later encoding run will overwrite the earlier one.'
    )

def resolve_artifact(root, exact_name, pattern):
    exact = Path(root) / exact_name
    if exact.exists():
        return exact
    matches = sorted(Path(root).glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"{root} does not contain a unique match for {pattern}: {[str(path) for path in matches]}"
        )
    return matches[0]

MANIFEST_PATH = resolve_artifact(
    ENCODING_ROOT,
    "xml_only_miditok_tsd24_topn_unk_v3_manifest.csv",
    "*manifest.csv",
)
VOCAB_PATH = resolve_artifact(
    ENCODING_ROOT,
    "xml_only_miditok_tsd24_topn_unk_v3_vocab.json",
    "*vocab.json",
)
MIDITOK_TOKENIZER_PATH = resolve_artifact(
    ENCODING_ROOT,
    "miditok_tsd24_topn_unk_v3_tokenizer.json",
    "*tokenizer.json",
)

EXPERIMENT_NAME = f"causal_lstm_tsd24_top{ACTIVE_TOP_N}_unk_v3_audited"
EXPERIMENT_ROOT = OUTPUT_ROOT / EXPERIMENT_NAME
WINDOW_ROOT = EXPERIMENT_ROOT / "windows_t512_trainh256_evalh512"
REPORT_ROOT = EXPERIMENT_ROOT / "report_assets"
COMPARISON_ROOT = OUTPUT_ROOT / "causal_lstm_topn_comparison_v3"
for directory in [EXPERIMENT_ROOT, WINDOW_ROOT, REPORT_ROOT, COMPARISON_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

WINDOW_TOKENS = 512 
TRAIN_HOP_TOKENS = 256
EVAL_HOP_TOKENS = 512
MIN_WINDOW_TOKENS = 128

EMBEDDING_DIM = 128
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.30

BATCH_SIZE = 16
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
USE_CLASS_WEIGHTS = False
RUN_OVERFIT_SMOKE_TEST = True

CAUSALITY_ATOL = 1e-4
LATENCY_WARMUP_STEPS = 50
LATENCY_MEASURE_STEPS = 500


MAX_TRAIN_BATCHES = None
MAX_VALIDATION_BATCHES = None

print("Active Top-N:", ACTIVE_TOP_N)
print("Encoding root:", ENCODING_ROOT)
print("Experiment root:", EXPERIMENT_ROOT)
print("Expected output classes including PAD:", EXPECTED_CHORD_VOCAB_SIZE)

## Encoding-interface preflight

The checks confirm the requested Top-N identity, the `<PAD> + <UNK> + Top-N` vocabulary, training-only label selection, valid supervised targets, mapped chord-onset reconstruction, shard consistency and the absence of group leakage.


In [ ]:
required_artifact_paths = {
    "manifest": MANIFEST_PATH,
    "summary": SUMMARY_PATH,
    "vocabulary": VOCAB_PATH,
    "MIDITok tokenizer": MIDITOK_TOKENIZER_PATH,
    "shard directory": SHARD_DIR,
}
missing_paths = [
    f"{name}: {path}"
    for name, path in required_artifact_paths.items()
    if not path.exists()
]
if missing_paths:
    raise FileNotFoundError(
        "Missing completed encoding artifacts:\n- " + "\n- ".join(missing_paths)
    )

with SUMMARY_PATH.open(encoding="utf-8") as handle:
    encoding_summary = json.load(handle)
with VOCAB_PATH.open(encoding="utf-8") as handle:
    vocab_payload = json.load(handle)
manifest_df = pd.read_csv(MANIFEST_PATH)

expected_summary_values = {
    "preprocessing_revision": EXPECTED_PREPROCESSING_REVISION,
    "data_source": "single_mxl_only",
    "jams_used": False,
    "ppq": EXPECTED_PPQ,
    "miditok_used": True,
    "miditok_version": EXPECTED_MIDITOK_VERSION,
    "symusic_version": EXPECTED_SYMUSIC_VERSION,
    "tokenizer": "TSD",
    "miditok_use_chords": False,
    "chord_reduction_strategy": "top_n_to_unk",
    "vocabulary_selected_from_split": "train",
    "requested_top_n": ACTIVE_TOP_N,
    "retained_chord_label_count": EXPECTED_RETAINED_CHORD_COUNT,
    "supervised_chord_class_count_including_unk": (
        EXPECTED_SUPERVISED_CHORD_CLASSES
    ),
    "chord_vocab_size_including_pad": EXPECTED_CHORD_VOCAB_SIZE,
    "gold_chord_token_inside_input_count": 0,
    "song_count": EXPECTED_SONG_COUNT,
    "one_supervised_slot_per_grid_step": True,
    "song_group_leakage_count": 0,
    "input_pad_inside_song_count": 0,
    "unknown_chord_target_allowed": True,
    "onset_recomputed_after_mapping": True,
    "array_length_mismatch_count": 0,
    "pending_count": 0,
    "model_input_array": "input_ids",
    "model_target_array": "chord_target_ids",
    "model_loss_mask_array": "grid_mask",
    "ready_for_lstm_windows": True,
}
summary_mismatches = {
    key: {"expected": expected, "actual": encoding_summary.get(key)}
    for key, expected in expected_summary_values.items()
    if encoding_summary.get(key) != expected
}
if summary_mismatches:
    raise RuntimeError(f"Encoding summary mismatch: {summary_mismatches}")

encoding_revision = str(encoding_summary.get("encoding_revision", ""))
if not (
    encoding_revision.startswith("XML_ONLY_MIDITOK_TSD24_")
    and "_UNK_V3_" in encoding_revision
):
    raise RuntimeError(
        f"Unexpected Top-N encoding revision: {encoding_revision}"
    )

installed_versions = {
    "miditok": importlib.metadata.version("miditok"),
    "symusic": importlib.metadata.version("symusic"),
}
expected_installed_versions = {
    "miditok": EXPECTED_MIDITOK_VERSION,
    "symusic": EXPECTED_SYMUSIC_VERSION,
}
if installed_versions != expected_installed_versions:
    raise RuntimeError(
        f"Tokenizer dependency mismatch: installed={installed_versions}, "
        f"expected={expected_installed_versions}"
    )

if vocab_payload.get("preprocessing_revision") != EXPECTED_PREPROCESSING_REVISION:
    raise RuntimeError("Vocabulary preprocessing revision mismatch")
if vocab_payload.get("encoding_revision") != encoding_revision:
    raise RuntimeError("Vocabulary encoding revision mismatch")

required_manifest_columns = {
    "record_id", "source_index", "relative_path", "file_name", "group_id",
    "split", "shard_file", "shard_song_index", "offset_start", "offset_end",
    "aligned_token_count", "raw_offset_start", "raw_offset_end",
    "raw_token_count", "grid_steps", "melody_event_count",
    "chord_interval_count", "mapped_chord_onset_count",
    "unk_chord_onset_count", "unk_grid_step_count", "unk_grid_rate",
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(f"Manifest columns missing: {missing_manifest_columns}")
if len(manifest_df) != EXPECTED_SONG_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_SONG_COUNT} encoded songs, found {len(manifest_df)}"
    )
if not manifest_df["record_id"].astype(str).is_unique:
    raise RuntimeError("Encoding manifest record_id is not unique")
if set(manifest_df["split"].astype(str)) != {"train", "validation", "test"}:
    raise RuntimeError("Manifest split names do not match train/validation/test")
if manifest_df.groupby("group_id")["split"].nunique().max() != 1:
    raise RuntimeError("A song group appears in multiple splits")
if not np.array_equal(
    manifest_df["offset_end"].to_numpy()
    - manifest_df["offset_start"].to_numpy(),
    manifest_df["aligned_token_count"].to_numpy(),
):
    raise RuntimeError("Manifest aligned offsets do not match token counts")

miditok_info = vocab_payload["miditok"]
chord_info = vocab_payload["chord_target"]
SELECTED_PPQ = int(vocab_payload["ppq"])
MIDITOK_VOCAB_SIZE = int(miditok_info["vocab_size"])
MIDITOK_PAD_ID = int(miditok_info["pad_token_id"])
MIDITOK_BOS_ID = int(miditok_info["bos_token_id"])
MIDITOK_EOS_ID = int(miditok_info["eos_token_id"])
MIDITOK_CLOCK_ID = int(miditok_info["clock_token_id"])
if SELECTED_PPQ != EXPECTED_PPQ:
    raise RuntimeError(f"Expected {EXPECTED_PPQ} PPQ, found {SELECTED_PPQ}")
if MIDITOK_VOCAB_SIZE != EXPECTED_MIDITOK_VOCAB_SIZE:
    raise RuntimeError("MIDITok vocabulary size mismatch")

NOTE_OFF_ID = MIDITOK_VOCAB_SIZE
MODEL_INPUT_VOCAB_SIZE = MIDITOK_VOCAB_SIZE + 1
CHORD_PAD_ID = int(chord_info["PAD"])
CHORD_UNK_ID = int(chord_info["UNK"])
CHORD_VOCAB_SIZE = int(chord_info["vocab_size"])
CHORD_TO_ID = {
    str(key): int(value)
    for key, value in chord_info["label_to_token"].items()
}
ID_TO_CHORD = {
    int(key): str(value)
    for key, value in chord_info["token_to_label"].items()
}

if CHORD_VOCAB_SIZE != EXPECTED_CHORD_VOCAB_SIZE:
    raise RuntimeError(
        f"Expected chord vocabulary {EXPECTED_CHORD_VOCAB_SIZE}, "
        f"found {CHORD_VOCAB_SIZE}"
    )
if CHORD_TO_ID.get("<PAD>") != 0 or CHORD_TO_ID.get("<UNK>") != 1:
    raise RuntimeError("Chord PAD/UNK mapping is inconsistent")
if CHORD_PAD_ID != 0 or CHORD_UNK_ID != 1:
    raise RuntimeError("Expected chord PAD=0 and UNK=1")
if len(CHORD_TO_ID) != CHORD_VOCAB_SIZE:
    raise RuntimeError("Chord label_to_token size mismatch")
if set(CHORD_TO_ID.values()) != set(range(CHORD_VOCAB_SIZE)):
    raise RuntimeError("Chord target IDs are not contiguous")
if int(chord_info.get("retained_label_count", -1)) != ACTIVE_TOP_N:
    raise RuntimeError("Vocabulary retained label count mismatch")

miditok_tokenizer = TSD(params=MIDITOK_TOKENIZER_PATH)
if len(miditok_tokenizer) != MIDITOK_VOCAB_SIZE:
    raise RuntimeError("Loaded MIDITok vocabulary size mismatch")
if str(miditok_tokenizer[MIDITOK_CLOCK_ID]) != str(
    miditok_info["clock_token"]
):
    raise RuntimeError("MIDITok clock token ID did not round-trip")
if str(miditok_info["clock_token"]) != "TimeShift_0.1.24":
    raise RuntimeError("Loaded clock token is not the 24-PPQ clock")

DURATION_TOKEN_IDS = np.asarray(
    miditok_tokenizer.token_ids_of_type("Duration"), dtype=np.int64
)
if len(DURATION_TOKEN_IDS) == 0:
    raise RuntimeError("No MIDITok Duration tokens were found")

expected_shard_keys = {
    "input_ids", "event_steps", "chord_target_ids", "chord_onset",
    "grid_mask", "raw_input_ids", "raw_event_steps", "offsets",
    "raw_offsets", "record_ids", "splits",
}
shard_names = manifest_df["shard_file"].astype(str)
if any(Path(name).name != name for name in shard_names):
    raise RuntimeError("Manifest shard_file must contain filenames")
unique_shard_names = shard_names.drop_duplicates().tolist()
missing_shards = [
    name for name in unique_shard_names
    if not (SHARD_DIR / name).is_file()
]
if missing_shards:
    raise FileNotFoundError(
        f"Referenced shard files are missing: {missing_shards[:10]}"
    )

total_validated_grid_slots = 0
total_validated_unk_slots = 0
for shard_name in tqdm(unique_shard_names, desc="Audit Top-N encoding shards"):
    shard_rows = (
        manifest_df.loc[manifest_df["shard_file"].astype(str) == shard_name]
        .sort_values("shard_song_index")
        .reset_index(drop=True)
    )
    with np.load(SHARD_DIR / shard_name, allow_pickle=False) as shard:
        missing_keys = sorted(expected_shard_keys - set(shard.files))
        if missing_keys:
            raise RuntimeError(f"Shard {shard_name} missing keys: {missing_keys}")
        shard_record_ids = shard["record_ids"].astype(str)
        shard_splits = shard["splits"].astype(str)
        shard_offsets = shard["offsets"].astype(np.int64)
        shard_raw_offsets = shard["raw_offsets"].astype(np.int64)
        expected_indices = np.arange(len(shard_rows))
        if not np.array_equal(
            shard_rows["shard_song_index"].to_numpy(dtype=int),
            expected_indices,
        ):
            raise RuntimeError(f"Non-contiguous shard indices in {shard_name}")
        if shard_record_ids.tolist() != (
            shard_rows["record_id"].astype(str).tolist()
        ):
            raise RuntimeError(f"Manifest/shard record mismatch in {shard_name}")
        if shard_splits.tolist() != shard_rows["split"].astype(str).tolist():
            raise RuntimeError(f"Manifest/shard split mismatch in {shard_name}")
        if len(shard_offsets) != len(shard_rows) + 1:
            raise RuntimeError(f"Aligned offset count mismatch in {shard_name}")
        if len(shard_raw_offsets) != len(shard_rows) + 1:
            raise RuntimeError(f"Raw offset count mismatch in {shard_name}")

        for song_index, row in shard_rows.iterrows():
            start = int(shard_offsets[song_index])
            end = int(shard_offsets[song_index + 1])
            if start != int(row["offset_start"]) or end != int(row["offset_end"]):
                raise RuntimeError(f"Manifest/shard offsets mismatch in {shard_name}")

            inputs = shard["input_ids"][start:end]
            steps = shard["event_steps"][start:end]
            targets = shard["chord_target_ids"][start:end]
            mask = shard["grid_mask"][start:end].astype(bool)
            onset = shard["chord_onset"][start:end].astype(bool)
            if len(inputs) != int(row["aligned_token_count"]):
                raise RuntimeError("Aligned token count mismatch")
            if int(inputs[0]) != MIDITOK_BOS_ID or int(inputs[-1]) != MIDITOK_EOS_ID:
                raise RuntimeError("BOS/EOS mismatch")
            if MIDITOK_PAD_ID in inputs:
                raise RuntimeError("Input PAD found inside encoded song")
            if not np.all(inputs[mask] == MIDITOK_CLOCK_ID):
                raise RuntimeError("Supervised input is not MIDITok clock")
            if not np.array_equal(steps[mask], np.arange(int(row["grid_steps"]))):
                raise RuntimeError("Clock steps are not contiguous")
            if np.any(targets[mask] == CHORD_PAD_ID):
                raise RuntimeError("Supervised target contains PAD")
            if targets[mask].min() < CHORD_UNK_ID:
                raise RuntimeError("Supervised target below UNK ID")
            if targets[mask].max() >= CHORD_VOCAB_SIZE:
                raise RuntimeError("Supervised target exceeds vocabulary")
            if np.any(targets[~mask] != CHORD_PAD_ID):
                raise RuntimeError("Gold chord target outside grid mask")
            if np.any(onset[~mask]):
                raise RuntimeError("Chord onset outside grid mask")

            grid_targets = targets[mask]
            grid_onsets = onset[mask]
            expected_onsets = np.zeros(len(grid_targets), dtype=bool)
            expected_onsets[0] = True
            if len(grid_targets) > 1:
                expected_onsets[1:] = grid_targets[1:] != grid_targets[:-1]
            if not np.array_equal(grid_onsets, expected_onsets):
                raise RuntimeError("Onsets were not rebuilt after Top-N mapping")

            song_unk_slots = int((grid_targets == CHORD_UNK_ID).sum())
            if song_unk_slots != int(row["unk_grid_step_count"]):
                raise RuntimeError("Song UNK slot count mismatch")
            total_validated_grid_slots += len(grid_targets)
            total_validated_unk_slots += song_unk_slots

if total_validated_grid_slots != int(
    encoding_summary["supervised_grid_slot_count"]
):
    raise RuntimeError("Global supervised grid count mismatch")
if total_validated_unk_slots != int(
    encoding_summary["unknown_chord_on_supervised_slot_count"]
):
    raise RuntimeError("Global UNK target count mismatch")

def load_encoded_song_arrays(row):
    start = int(row["offset_start"])
    end = int(row["offset_end"])
    with np.load(SHARD_DIR / str(row["shard_file"]), allow_pickle=False) as shard:
        return {
            "input_ids": shard["input_ids"][start:end],
            "event_steps": shard["event_steps"][start:end],
            "chord_target_ids": shard["chord_target_ids"][start:end],
            "grid_mask": shard["grid_mask"][start:end],
            "chord_onset": shard["chord_onset"][start:end],
        }

interface_sample_rows = []
for split_name in ["train", "validation", "test"]:
    row = manifest_df.loc[manifest_df["split"] == split_name].iloc[0].to_dict()
    arrays = load_encoded_song_arrays(row)
    sample_mask = arrays["grid_mask"].astype(bool)
    interface_sample_rows.append(
        {
            "split": split_name,
            "record_id": str(row["record_id"]),
            "shard_file": str(row["shard_file"]),
            "aligned_tokens": int(len(arrays["input_ids"])),
            "grid_steps": int(sample_mask.sum()),
            "unk_targets": int(
                (arrays["chord_target_ids"][sample_mask] == CHORD_UNK_ID).sum()
            ),
            "status": "PASS",
        }
    )

test_record_ids = sorted(
    manifest_df.loc[
        manifest_df["split"].eq("test"), "record_id"
    ].astype(str).tolist()
)
TEST_SPLIT_SHA256 = hashlib.sha256(
    "|".join(test_record_ids).encode("utf-8")
).hexdigest()



split_assignment_text = "|".join(
    f"{row.record_id}:{row.split}"
    for row in manifest_df[["record_id", "split"]]
    .astype(str)
    .sort_values("record_id")
    .itertuples(index=False)
)
SPLIT_ASSIGNMENT_SHA256 = hashlib.sha256(
    split_assignment_text.encode("utf-8")
).hexdigest()

input_interface_columns = [
    "record_id",
    "group_id",
    "split",
    "aligned_token_count",
    "grid_steps",
    "melody_event_count",
]
input_interface_text = manifest_df[input_interface_columns].sort_values(
    "record_id"
).to_csv(index=False, lineterminator="\n")
INPUT_INTERFACE_SHA256 = hashlib.sha256(
    input_interface_text.encode("utf-8")
).hexdigest()

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

ENCODING_ARTIFACT_SHA256 = {
    "summary": sha256_file(SUMMARY_PATH),
    "vocabulary": sha256_file(VOCAB_PATH),
    "manifest": sha256_file(MANIFEST_PATH),
}

interface_preflight = {
    "schema_revision": COMPATIBILITY_SCHEMA_REVISION,
    "status": "ENCODING_INTERFACE_PASS",
    "active_top_n": ACTIVE_TOP_N,
    "encoding_root": str(ENCODING_ROOT),
    "encoding_revision": encoding_revision,
    "song_count": int(len(manifest_df)),
    "split_song_counts": {
        str(key): int(value)
        for key, value in manifest_df["split"].value_counts().to_dict().items()
    },
    "test_split_sha256": TEST_SPLIT_SHA256,
    "split_assignment_sha256": SPLIT_ASSIGNMENT_SHA256,
    "input_interface_sha256": INPUT_INTERFACE_SHA256,
    "encoding_artifact_sha256": ENCODING_ARTIFACT_SHA256,
    "shard_count": int(len(unique_shard_names)),
    "miditok_vocab_size": MIDITOK_VOCAB_SIZE,
    "chord_vocab_size_including_pad": CHORD_VOCAB_SIZE,
    "supervised_chord_classes": EXPECTED_SUPERVISED_CHORD_CLASSES,
    "unk_target_count": total_validated_unk_slots,
    "unk_target_rate": (
        total_validated_unk_slots / total_validated_grid_slots
    ),
    "samples": interface_sample_rows,
}
with (EXPERIMENT_ROOT / "compatibility_preflight.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(interface_preflight, handle, indent=2)

display(pd.DataFrame(interface_sample_rows))
print("ENCODING INTERFACE PREFLIGHT PASSED")
print("Active Top-N:", ACTIVE_TOP_N)
print("Encoding root:", ENCODING_ROOT)
print("Encoded songs:", len(manifest_df))
print("Chord vocabulary including PAD:", CHORD_VOCAB_SIZE)
print("Supervised classes retained + UNK:", EXPECTED_SUPERVISED_CHORD_CLASSES)
print(
    "All-data UNK target rate:",
    f"{100 * interface_preflight['unk_target_rate']:.4f}%",
)

## Reduced-label dataset statistics and chord n-grams

N-grams are calculated from onsets recomputed after vocabulary reduction. `<UNK>` is retained in audit statistics but excluded from musical-progression displays.


In [ ]:
def ngrams(items, n):
    return zip(*(items[offset:] for offset in range(n)))

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for shard_file, shard_rows in tqdm(
    manifest_df.groupby("shard_file", sort=False),
    desc="Mapped chord event statistics",
):
    with np.load(SHARD_DIR / shard_file, allow_pickle=False) as shard:
        shard_targets = shard["chord_target_ids"]
        shard_onsets = shard["chord_onset"]
        shard_masks = shard["grid_mask"]
        for row in shard_rows.to_dict("records"):
            start = int(row["offset_start"])
            end = int(row["offset_end"])
            targets = shard_targets[start:end]
            onset = shard_onsets[start:end].astype(bool)
            mask = shard_masks[start:end].astype(bool)
            ids = targets[onset & mask].astype(int).tolist()
            labels = [ID_TO_CHORD[token_id] for token_id in ids]
            unigram_counts.update(labels)
            bigram_counts.update(ngrams(labels, 2))
            trigram_counts.update(ngrams(labels, 3))

def counter_frame(counter, n, top_k=20, exclude_unk=False):
    rows = []
    for key, count in counter.most_common():
        parts = (key,) if n == 1 else key
        if exclude_unk and "<UNK>" in parts:
            continue
        label = key if n == 1 else " -> ".join(key)
        rows.append({"n": n, "progression": label, "count": count})
        if len(rows) >= top_k:
            break
    return pd.DataFrame(rows)

ngram_stats_df = pd.concat(
    [
        counter_frame(unigram_counts, 1),
        counter_frame(bigram_counts, 2),
        counter_frame(trigram_counts, 3),
    ],
    ignore_index=True,
)
musical_ngram_stats_df = pd.concat(
    [
        counter_frame(unigram_counts, 1, exclude_unk=True),
        counter_frame(bigram_counts, 2, exclude_unk=True),
        counter_frame(trigram_counts, 3, exclude_unk=True),
    ],
    ignore_index=True,
)
ngram_stats_df.to_csv(
    EXPERIMENT_ROOT / "chord_ngram_stats_including_unk.csv", index=False
)
musical_ngram_stats_df.to_csv(
    EXPERIMENT_ROOT / "chord_ngram_stats_excluding_unk.csv", index=False
)

dataset_stats = pd.Series(
    {
        "active_top_n": ACTIVE_TOP_N,
        "encoded_songs": len(manifest_df),
        "median_melody_events_per_song": (
            manifest_df["melody_event_count"].median()
        ),
        "median_raw_chord_intervals_per_song": (
            manifest_df["chord_interval_count"].median()
        ),
        "median_mapped_chord_onsets_per_song": (
            manifest_df["mapped_chord_onset_count"].median()
        ),
        "median_24ppq_grid_steps_per_song": (
            manifest_df["grid_steps"].median()
        ),
        "chord_vocabulary_including_pad": CHORD_VOCAB_SIZE,
        "mapped_onset_labels_including_unk": len(unigram_counts),
        "mapped_unk_onset_count": int(unigram_counts.get("<UNK>", 0)),
    },
    name="value",
)
display(dataset_stats.to_frame())
display(ngram_stats_df[ngram_stats_df["n"] == 1].head(15))
display(musical_ngram_stats_df[musical_ngram_stats_df["n"] == 3].head(15))

top_chords = musical_ngram_stats_df[
    musical_ngram_stats_df["n"] == 1
].head(12)
plt.figure(figsize=(10, 5))
plt.barh(top_chords["progression"][::-1], top_chords["count"][::-1])
plt.xlabel("Mapped chord-onset occurrences")
plt.title(f"Top {ACTIVE_TOP_N}: most frequent retained chord labels")
plt.tight_layout()
plt.savefig(REPORT_ROOT / "top_chords.png", dpi=180, bbox_inches="tight")
plt.show()

## Construct causal windows without crossing song boundaries

Offline duration tokens are used only to schedule release-time `NOTE_OFF` events and are removed from live model input. A clock token remains at every 24-PPQ step, and chord targets are attached only to clock positions. Training windows overlap; validation and test windows are non-overlapping and retain state within each song.


In [ ]:
LIVE_INPUT_SCHEMA = {
    "representation": "MIDITok-TSD-derived causal live event stream",
    "events": ["Pitch_at_note_on", "NOTE_OFF_at_release", "TimeShift_0.1.24_clock"],
    "offline_duration_visible_to_model": False,
    "note_off_token_id": NOTE_OFF_ID,
}

WINDOW_SCHEMA = {
    "schema_revision": COMPATIBILITY_SCHEMA_REVISION,
    "encoding_revision": encoding_summary["encoding_revision"],
    "encoding_artifact_sha256": ENCODING_ARTIFACT_SHA256,
    "active_top_n": ACTIVE_TOP_N,
    "chord_vocab_size": CHORD_VOCAB_SIZE,
    "unk_target_allowed": True,
    "onset_recomputed_after_mapping": True,
    "encoding_song_count": int(encoding_summary["song_count"]),
    "aligned_miditok_token_count": int(encoding_summary["aligned_miditok_token_count"]),
    "live_input_schema": LIVE_INPUT_SCHEMA,
    "window_tokens": WINDOW_TOKENS,
    "train_hop_tokens": TRAIN_HOP_TOKENS,
    "eval_hop_tokens": EVAL_HOP_TOKENS,
    "min_window_tokens": MIN_WINDOW_TOKENS,
    "keep_all_validation_test_tail_tokens": True,
}

def duration_token_to_steps(token_id):
    token = str(miditok_tokenizer[int(token_id)])
    token_type, value = token.split("_", 1)
    if token_type != "Duration":
        raise ValueError(f"Expected Duration token, got {token}")
    beats, samples, resolution = [int(part) for part in value.split(".")]
    steps = beats * SELECTED_PPQ + int(round(samples * SELECTED_PPQ / resolution))
    if steps <= 0:
        raise ValueError(f"Non-positive Duration token: {token}")
    return steps

def build_causal_song_arrays(
    song_inputs,
    song_event_steps,
    song_targets,
    song_masks,
    song_onsets,
):
    lengths = {
        len(song_inputs),
        len(song_event_steps),
        len(song_targets),
        len(song_masks),
        len(song_onsets),
    }
    if len(lengths) != 1:
        raise ValueError("Encoded song arrays have unequal lengths")

    grid_indices = np.flatnonzero(song_masks.astype(bool))
    grid_steps = len(grid_indices)
    if not np.array_equal(song_event_steps[grid_indices], np.arange(grid_steps)):
        raise ValueError("Clock event_steps are not exactly 0..grid_steps-1")
    if not np.all(song_inputs[grid_indices] == MIDITOK_CLOCK_ID):
        raise ValueError("A supervised position is not the MIDITok clock")

    pitches_by_step = {}
    durations_by_step = {}
    allowed_non_grid_types = {"BOS", "EOS", "Pitch", "Duration"}
    for token_id, event_step, is_grid in zip(
        song_inputs, song_event_steps, song_masks
    ):
        if is_grid:
            continue
        token_id = int(token_id)
        event_step = int(event_step)
        token_type = str(miditok_tokenizer.token_id_type(token_id))
        if token_type not in allowed_non_grid_types:
            raise ValueError(
                f"Unexpected token type in aligned song: {token_type}"
            )
        if token_type == "Pitch":
            pitches_by_step.setdefault(event_step, []).append(token_id)
        elif token_type == "Duration":
            durations_by_step.setdefault(event_step, []).append(token_id)

    note_offs_by_step = Counter()
    note_count = 0
    for step in sorted(set(pitches_by_step) | set(durations_by_step)):
        pitch_ids = pitches_by_step.get(step, [])
        duration_ids = durations_by_step.get(step, [])
        if len(pitch_ids) != len(duration_ids):
            raise ValueError(
                f"Pitch/Duration count mismatch at step {step}: "
                f"{len(pitch_ids)} != {len(duration_ids)}"
            )
        for duration_id in duration_ids: 
            end_step = step + duration_token_to_steps(duration_id)
            if not 0 < end_step <= grid_steps:
                raise ValueError(
                    f"NOTE_OFF outside song: onset={step}, end={end_step}, "
                    f"grid_steps={grid_steps}"
                )
            note_offs_by_step[end_step] += 1
            note_count += 1

    causal_inputs = [MIDITOK_BOS_ID]
    causal_targets = [CHORD_PAD_ID]
    causal_masks = [0]
    causal_onsets = [0]

    def append_unsupervised(token_id):
        causal_inputs.append(int(token_id))
        causal_targets.append(CHORD_PAD_ID)
        causal_masks.append(0)
        causal_onsets.append(0)

    for step in range(grid_steps): 
        # When release and attack share a timestamp, close the old note first.
        for _ in range(note_offs_by_step.get(step, 0)):
            append_unsupervised(NOTE_OFF_ID)
        for pitch_id in pitches_by_step.get(step, []): 
            append_unsupervised(pitch_id)

        original_index = int(grid_indices[step])
        causal_inputs.append(MIDITOK_CLOCK_ID) 
        causal_targets.append(int(song_targets[original_index]))
        causal_masks.append(1)
        causal_onsets.append(int(song_onsets[original_index]))

    # A note can be released exactly at the end of the final grid slot.
    for _ in range(note_offs_by_step.get(grid_steps, 0)):
        append_unsupervised(NOTE_OFF_ID)
    causal_inputs.append(MIDITOK_EOS_ID)
    causal_targets.append(CHORD_PAD_ID)
    causal_masks.append(0)
    causal_onsets.append(0)

    arrays = {
        "input_ids": np.asarray(causal_inputs, dtype=np.int32),
        "chord_target_ids": np.asarray(causal_targets, dtype=np.uint16),
        "grid_mask": np.asarray(causal_masks, dtype=np.uint8),
        "chord_onset": np.asarray(causal_onsets, dtype=np.uint8),
    }
    if len(arrays["input_ids"]) != len(song_inputs):
        raise ValueError(
            "Causal conversion changed sequence length: "
            f"{len(song_inputs)} -> {len(arrays['input_ids'])}"
        )
    if int((arrays["input_ids"] == NOTE_OFF_ID).sum()) != note_count:
        raise ValueError("NOTE_OFF count does not match melody note count")
    if int(arrays["grid_mask"].sum()) != grid_steps:
        raise ValueError("Causal conversion changed supervised grid count")
    if np.any(np.isin(arrays["input_ids"], DURATION_TOKEN_IDS)): 
        raise ValueError("A future-revealing Duration token remains in model input")
    if np.any(arrays["chord_target_ids"][arrays["grid_mask"] == 0] != CHORD_PAD_ID):
        raise ValueError("Gold chord target leaked into an unsupervised event")
    return arrays

def starts_for_song(sequence_length, hop_tokens, keep_short_tail=False): 
    starts = []
    for start in range(0, int(sequence_length), hop_tokens):
        end = min(start + WINDOW_TOKENS, int(sequence_length))
        if end - start < MIN_WINDOW_TOKENS and not keep_short_tail:
            break
        starts.append((start, end))
        if end == int(sequence_length):
            break
    return starts

def window_paths(split_name):
    hop_tokens = TRAIN_HOP_TOKENS if split_name == "train" else EVAL_HOP_TOKENS
    stem = f"{split_name}_t{WINDOW_TOKENS}_h{hop_tokens}"
    return WINDOW_ROOT / f"{stem}.npz", WINDOW_ROOT / f"{stem}_done.json"

def window_artifact_valid(split_name):
    npz_path, done_path = window_paths(split_name)
    if not npz_path.exists() or not done_path.exists():
        return False
    try:
        marker = json.loads(done_path.read_text(encoding="utf-8"))
        return marker["schema"] == WINDOW_SCHEMA and marker["split"] == split_name
    except Exception:
        return False

def save_npz_atomic(path, **arrays):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)

def build_windows(split_name):
    split_rows = manifest_df[manifest_df["split"] == split_name].copy()
    hop_tokens = TRAIN_HOP_TOKENS if split_name == "train" else EVAL_HOP_TOKENS
    total_windows = sum(
        len(
            starts_for_song(
                row.aligned_token_count,
                hop_tokens,
                keep_short_tail=split_name != "train",
            )
        )
        for row in split_rows.itertuples(index=False)
    )
    if total_windows == 0:
        raise RuntimeError(f"No windows for split {split_name}")

    inputs = np.full(
        (total_windows, WINDOW_TOKENS), MIDITOK_PAD_ID, dtype=np.int32
    )
    targets = np.full(
        (total_windows, WINDOW_TOKENS), CHORD_PAD_ID, dtype=np.uint16
    )
    masks = np.zeros((total_windows, WINDOW_TOKENS), dtype=np.uint8)
    onsets = np.zeros((total_windows, WINDOW_TOKENS), dtype=np.uint8)
    lengths = np.zeros(total_windows, dtype=np.uint16)
    record_ids = np.empty(total_windows, dtype="<U64")
    token_starts = np.zeros(total_windows, dtype=np.int32)

    cursor = 0
    total_note_offs = 0
    for shard_file, shard_rows in tqdm(
        split_rows.groupby("shard_file", sort=False),
        desc=f"Build {split_name} causal windows",
    ):
        shard_path = SHARD_DIR / shard_file
        with np.load(shard_path, allow_pickle=False) as shard:
            for row in shard_rows.to_dict("records"): 
                song_start = int(row["offset_start"])
                song_end = int(row["offset_end"])
                causal = build_causal_song_arrays(
                    shard["input_ids"][song_start:song_end],
                    shard["event_steps"][song_start:song_end],
                    shard["chord_target_ids"][song_start:song_end],
                    shard["grid_mask"][song_start:song_end],
                    shard["chord_onset"][song_start:song_end],
                )
                song_length = len(causal["input_ids"])
                if song_length != int(row["aligned_token_count"]):
                    raise ValueError("Manifest aligned_token_count mismatch")
                total_note_offs += int((causal["input_ids"] == NOTE_OFF_ID).sum())

                for local_start, local_end in starts_for_song(
                    song_length,
                    hop_tokens,
                    keep_short_tail=split_name != "train",
                ):
                    length = local_end - local_start
                    inputs[cursor, :length] = causal["input_ids"][local_start:local_end]
                    targets[cursor, :length] = causal["chord_target_ids"][local_start:local_end]
                    masks[cursor, :length] = causal["grid_mask"][local_start:local_end]
                    onsets[cursor, :length] = causal["chord_onset"][local_start:local_end]
                    lengths[cursor] = length
                    record_ids[cursor] = str(row["record_id"])
                    token_starts[cursor] = local_start
                    cursor += 1

    if cursor != total_windows:
        raise RuntimeError(
            f"Window count mismatch for {split_name}: "
            f"built={cursor}, expected={total_windows}"
        )

    supervised_slots_per_window = masks.sum(axis=1)
    zero_supervision_indices = np.flatnonzero(
        supervised_slots_per_window == 0
    )
    zero_supervision_tail_window_count = int(
        len(zero_supervision_indices)
    )
    if split_name == "train" and zero_supervision_tail_window_count:
        raise RuntimeError(
            "A train window contains no supervised chord slot; "
            f"first window index={int(zero_supervision_indices[0])}"
        )
    if split_name != "train" and zero_supervision_tail_window_count:
        non_final_zero_indices = [
            int(index)
            for index in zero_supervision_indices
            if index + 1 < total_windows
            and record_ids[index + 1] == record_ids[index]
        ]
        if non_final_zero_indices:
            raise RuntimeError(
                f"{split_name} has an event-only window before the "
                f"end of its song: {non_final_zero_indices[:10]}"
            )
        print(
            f"{split_name}: keeping "
            f"{zero_supervision_tail_window_count} event-only tail "
            "window(s) for complete stateful streaming"
        )

    if not np.all(targets[masks == 0] == CHORD_PAD_ID):
        raise RuntimeError(
            f"Gold chord target leaked outside the mask in {split_name}"
        )
    supervised_targets = targets[masks == 1]
    if np.any(supervised_targets == CHORD_PAD_ID):
        raise RuntimeError(
            f"PAD chord target found on a supervised slot in {split_name}"
        )
    if supervised_targets.min() < CHORD_UNK_ID:
        raise RuntimeError(f"Target below UNK ID in {split_name}")
    if supervised_targets.max() >= CHORD_VOCAB_SIZE:
        raise RuntimeError(f"Target exceeds vocabulary in {split_name}")
    if np.any(np.isin(inputs[masks == 0], DURATION_TOKEN_IDS)):
        raise RuntimeError(
            f"A future-revealing Duration token remains in {split_name}"
        )

    npz_path, done_path = window_paths(split_name)
    save_npz_atomic(
        npz_path,
        input_ids=inputs,
        chord_target_ids=targets,
        grid_mask=masks,
        chord_onset=onsets,
        lengths=lengths,
        record_ids=record_ids,
        token_starts=token_starts,
    )
    done_path.write_text(
        json.dumps(
            {
                "schema": WINDOW_SCHEMA,
                "split": split_name,
                "song_count": int(len(split_rows)),
                "window_count": int(total_windows),
                "supervised_slot_count": int(masks.sum()),
                "unk_supervised_slot_count": int(
                    ((targets == CHORD_UNK_ID) & (masks == 1)).sum()
                ),
                "zero_supervision_tail_window_count": (
                    zero_supervision_tail_window_count
                ),
                "note_off_event_count_before_window_overlap": int(total_note_offs),
                "future_duration_tokens_in_model_input": 0,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    del inputs, targets, masks, onsets
    gc.collect()
    return npz_path

causal_adapter_sample_rows = []
for split_name in ["train", "validation", "test"]:
    row = manifest_df.loc[manifest_df["split"] == split_name].iloc[0].to_dict()
    original = load_encoded_song_arrays(row)
    causal = build_causal_song_arrays(
        original["input_ids"],
        original["event_steps"],
        original["chord_target_ids"],
        original["grid_mask"],
        original["chord_onset"],
    )
    note_off_count = int((causal["input_ids"] == NOTE_OFF_ID).sum())
    if note_off_count != int(row["melody_event_count"]):
        raise RuntimeError(
            f"NOTE_OFF count mismatch for {row['record_id']}: "
            f"{note_off_count} != {int(row['melody_event_count'])}"
        )
    if len(causal["input_ids"]) != int(row["aligned_token_count"]):
        raise RuntimeError(f"Causal sequence length mismatch for {row['record_id']}")
    causal_adapter_sample_rows.append(
        {
            "split": split_name,
            "record_id": str(row["record_id"]),
            "offline_tokens": int(row["aligned_token_count"]),
            "causal_tokens": int(len(causal["input_ids"])),
            "note_off_events": note_off_count,
            "duration_tokens_remaining": int(
                np.isin(causal["input_ids"], DURATION_TOKEN_IDS).sum()
            ),
            "supervised_slots": int(causal["grid_mask"].sum()),
            "unk_targets": int(
                (
                    causal["chord_target_ids"][
                        causal["grid_mask"].astype(bool)
                    ]
                    == CHORD_UNK_ID
                ).sum()
            ),
            "status": "PASS",
        }
    )

interface_preflight["status"] = "FULL_CAUSAL_INTERFACE_PASS"
interface_preflight["causal_adapter_samples"] = causal_adapter_sample_rows
with (EXPERIMENT_ROOT / "compatibility_preflight.json").open("w", encoding="utf-8") as handle:
    json.dump(interface_preflight, handle, indent=2)
display(pd.DataFrame(causal_adapter_sample_rows))
print("CAUSAL ADAPTER PREFLIGHT PASSED")
print("Both preflight gates passed; full window creation may begin.")

for split_name in ["train", "validation", "test"]:
    if window_artifact_valid(split_name):
        print("Reuse valid causal window artifact:", split_name)
    else:
        build_windows(split_name)

window_summary_rows = []
for split_name in ["train", "validation", "test"]:
    _, done_path = window_paths(split_name)
    marker = json.loads(done_path.read_text(encoding="utf-8"))
    marker.setdefault("zero_supervision_tail_window_count", 0)
    window_summary_rows.append(marker)
window_summary_df = pd.DataFrame(window_summary_rows)
display(
    window_summary_df[
        [
            "split",
            "song_count",
            "window_count",
            "supervised_slot_count",
            "zero_supervision_tail_window_count",
            "future_duration_tokens_in_model_input",
        ]
    ]
)

## PyTorch datasets and data loaders


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, split_name):
        npz_path, _ = window_paths(split_name)
        with np.load(npz_path, allow_pickle=False) as data:
            self.input_ids = data["input_ids"]
            self.targets = data["chord_target_ids"]
            self.masks = data["grid_mask"]
            self.onsets = data["chord_onset"]
            self.lengths = data["lengths"]
            self.record_ids = data["record_ids"].astype(str)
            self.token_starts = data["token_starts"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return {
            "input_ids": torch.as_tensor(self.input_ids[index], dtype=torch.long),
            "targets": torch.as_tensor(self.targets[index], dtype=torch.long),
            "mask": torch.as_tensor(self.masks[index], dtype=torch.bool),
            "onset": torch.as_tensor(self.onsets[index], dtype=torch.bool),
            "length": torch.tensor(int(self.lengths[index]), dtype=torch.long),
            "index": torch.tensor(index, dtype=torch.long),
        }

train_dataset = WindowDataset("train")
validation_dataset = WindowDataset("validation")
test_dataset = WindowDataset("test")

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "pin_memory": DEVICE.type == "cuda",
}
train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(RANDOM_SEED)
train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=train_loader_generator,
    **loader_kwargs,
)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print("Train windows:", len(train_dataset))
print("Validation windows:", len(validation_dataset))
print("Test windows:", len(test_dataset))
first_batch = next(iter(train_loader))
for key, value in first_batch.items():
    print(key, tuple(value.shape))

## Unidirectional LSTM and Top-N classification target

The architecture is a learned embedding, input dropout, two unidirectional LSTM layers, layer normalisation, dropout and a linear classifier. The final layer has 202 outputs for Top-200 or 252 outputs for Top-250, including `<PAD>` and `<UNK>`.


In [ ]:
class MelodyToChordCausalLSTM(nn.Module):
    def __init__(
        self,
        input_vocab_size,
        chord_vocab_size,
        pad_token_id,
        embedding_dim=128,
        hidden_size=128,
        num_layers=2,
        dropout=0.30,
    ):
        super().__init__()
        self.embedding = nn.Embedding( 
            input_vocab_size,
            embedding_dim,
            padding_idx=pad_token_id,
        )
        self.input_dropout = nn.Dropout(dropout) 
        self.lstm = nn.LSTM( 
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=False,
        )
        self.normalisation = nn.LayerNorm(hidden_size) 
        self.output_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, chord_vocab_size) 

    def forward(self, input_ids, state=None, return_state=False):
        embedded = self.input_dropout(self.embedding(input_ids))
        hidden, next_state = self.lstm(embedded, state)
        hidden = self.output_dropout(self.normalisation(hidden))
        logits = self.classifier(hidden) 
        if return_state:
            return logits, next_state
        return logits

    @torch.inference_mode()
    def stream_step(self, token_ids, state=None):
        '''Process only newly arrived token IDs and return reusable (h, c).'''
        if not torch.is_tensor(token_ids):
            token_ids = torch.as_tensor(token_ids, dtype=torch.long, device=next(self.parameters()).device)
        else:
            token_ids = token_ids.to(next(self.parameters()).device, dtype=torch.long)
        if token_ids.ndim == 0:
            token_ids = token_ids.view(1, 1)
        elif token_ids.ndim == 1:
            token_ids = token_ids.unsqueeze(1)
        elif token_ids.ndim != 2:
            raise ValueError("stream_step expects a scalar or [batch] token IDs")
        logits, next_state = self.forward(token_ids, state=state, return_state=True)
        return logits[:, -1, :], next_state

def make_model():
    return MelodyToChordCausalLSTM(
        MODEL_INPUT_VOCAB_SIZE,
        CHORD_VOCAB_SIZE,
        MIDITOK_PAD_ID,
        EMBEDDING_DIM,
        HIDDEN_SIZE,
        NUM_LAYERS,
        DROPOUT,
    )


train_targets = train_dataset.targets[
    train_dataset.masks.astype(bool)
].astype(int)
train_class_counts = np.bincount(
    train_targets, minlength=CHORD_VOCAB_SIZE
)
majority_chord_id = int(train_class_counts.argmax())
majority_chord_label = ID_TO_CHORD[majority_chord_id]
SUPERVISED_CHORD_CLASS_COUNT = CHORD_VOCAB_SIZE - 1

class_weights = np.ones(CHORD_VOCAB_SIZE, dtype=np.float32)
if USE_CLASS_WEIGHTS:
    seen = train_class_counts > 0
    class_weights[seen] = np.sqrt(
        train_class_counts[seen].max() / train_class_counts[seen]
    )
    class_weights[seen] /= class_weights[seen].mean()
    class_weights = np.clip(class_weights, 0.0, 5.0)
class_weights[CHORD_PAD_ID] = 0.0
loss_class_weights = torch.as_tensor(class_weights, device=DEVICE)

def masked_training_cross_entropy(logits, targets, mask):
    'Training objective with optional class weights; both baseline conditions are fixed to False.'
    per_token = F.cross_entropy(
        logits.transpose(1, 2),
        targets,
        weight=loss_class_weights,
        reduction="none",
    )
    valid = mask.float()
    return (per_token * valid).sum() / valid.sum().clamp_min(1.0)

def cross_entropy_sum_and_count(logits, targets, mask):
    'Unweighted likelihood used for reported cross-entropy and perplexity.'
    valid_logits = logits[mask]
    valid_targets = targets[mask]
    if valid_targets.numel() == 0:
        return 0.0, 0
    loss_sum = F.cross_entropy(
        valid_logits,
        valid_targets,
        reduction="sum",
    )
    return float(loss_sum), int(valid_targets.numel())

reset_random_state()
model = make_model().to(DEVICE)
parameter_count = sum(
    parameter.numel() for parameter in model.parameters()
)
train_unk_rate = float((train_targets == CHORD_UNK_ID).mean())

print(model)
print("Active Top-N:", ACTIVE_TOP_N)
print("Bidirectional:", model.lstm.bidirectional)
print("Trainable parameters:", f"{parameter_count:,}")
print("Training majority chord:", majority_chord_label, majority_chord_id)
print("Training UNK target rate:", f"{100 * train_unk_rate:.4f}%")
print("Class weights enabled:", USE_CLASS_WEIGHTS)

## Forward-pass, causality, streaming-equivalence and overfit checks

Future-perturbation testing verifies that later inputs cannot change earlier logits. Streaming-equivalence testing confirms that chunked inference with carried LSTM state reproduces full-sequence causal inference within numerical tolerance.


In [ ]:
@torch.inference_mode()
def run_causality_checks(model, sample_inputs, atol=CAUSALITY_ATOL):
    model.eval()
    sequence = sample_inputs[:1].to(DEVICE).clone()
    sequence_length = sequence.shape[1]
    cutoff = min(63, sequence_length - 2)

    altered = sequence.clone()
    altered[:, cutoff + 1 :] = (
        altered[:, cutoff + 1 :] + 17
    ) % MODEL_INPUT_VOCAB_SIZE

    original_logits = model(sequence)
    altered_logits = model(altered)
    future_max_abs_diff = float(
        (original_logits[:, : cutoff + 1] - altered_logits[:, : cutoff + 1])
        .abs()
        .max()
        .item()
    )

    state = None
    streamed_parts = []
    for start in range(0, sequence_length, 37):
        end = min(start + 37, sequence_length)
        part_logits, state = model(
            sequence[:, start:end], state=state, return_state=True
        )
        streamed_parts.append(part_logits)
    streamed_logits = torch.cat(streamed_parts, dim=1)
    streaming_max_abs_diff = float(
        (original_logits - streamed_logits).abs().max().item()
    )

    step_logits, step_state = model.stream_step(sequence[:, 0], state=None)
    expected_state_shape = (NUM_LAYERS, 1, HIDDEN_SIZE)
    if tuple(step_state[0].shape) != expected_state_shape:
        raise RuntimeError(
            f"Unexpected hidden state shape: {tuple(step_state[0].shape)}"
        )
    if tuple(step_state[1].shape) != expected_state_shape:
        raise RuntimeError(
            f"Unexpected cell state shape: {tuple(step_state[1].shape)}"
        )
    if tuple(step_logits.shape) != (1, CHORD_VOCAB_SIZE):
        raise RuntimeError(f"Unexpected stream-step logits: {tuple(step_logits.shape)}")

    result = {
        "bidirectional": bool(model.lstm.bidirectional),
        "future_perturbation_max_abs_diff": future_max_abs_diff,
        "streaming_equivalence_max_abs_diff": streaming_max_abs_diff,
        "tolerance": float(atol),
        "one_event_state_shape": list(expected_state_shape),
    }
    if result["bidirectional"]:
        raise RuntimeError("Real-time model is accidentally bidirectional")
    if future_max_abs_diff > atol:
        raise RuntimeError("Future tokens changed an earlier causal output")
    if streaming_max_abs_diff > atol:
        raise RuntimeError("Stateful streaming does not match one-pass inference")
    return result

model.eval()
with torch.no_grad():
    smoke_inputs = first_batch["input_ids"].to(DEVICE)
    smoke_targets = first_batch["targets"].to(DEVICE)
    smoke_mask = first_batch["mask"].to(DEVICE)
    smoke_logits = model(smoke_inputs)
    smoke_loss = masked_training_cross_entropy(smoke_logits, smoke_targets, smoke_mask)

assert smoke_logits.shape == (
    smoke_inputs.shape[0], WINDOW_TOKENS, CHORD_VOCAB_SIZE
)
initial_causality_checks = run_causality_checks(model, smoke_inputs)
print("Input shape:", tuple(smoke_inputs.shape))
print("Logit shape:", tuple(smoke_logits.shape))
print("Initial masked loss:", float(smoke_loss))
print("Initial causality checks:", initial_causality_checks)

if RUN_OVERFIT_SMOKE_TEST:
    smoke_subset = Subset(train_dataset, range(min(32, len(train_dataset))))
    smoke_loader = DataLoader(smoke_subset, batch_size=16, shuffle=True)
    smoke_model = make_model().to(DEVICE)
    smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=3e-3)
    smoke_losses = []
    for smoke_epoch in range(12):
        smoke_model.train()
        numerator = 0.0
        denominator = 0
        for batch in smoke_loader:
            inputs = batch["input_ids"].to(DEVICE)
            targets = batch["targets"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            smoke_optimizer.zero_grad(set_to_none=True)
            logits = smoke_model(inputs)
            loss = masked_training_cross_entropy(logits, targets, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), GRAD_CLIP)
            smoke_optimizer.step()
            valid = int(mask.sum())
            numerator += float(loss) * valid
            denominator += valid
        smoke_losses.append(numerator / max(1, denominator))
    print(
        "Overfit smoke loss:",
        round(smoke_losses[0], 4),
        "->",
        round(smoke_losses[-1], 4),
    )
    if smoke_losses[-1] >= smoke_losses[0]:
        raise RuntimeError("Overfit smoke test did not reduce loss")
    del smoke_model, smoke_optimizer, smoke_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## Training and stateful validation

Both vocabulary conditions use the same optimiser, batch size, seed, windowing and early-stopping protocol. Checkpoint selection uses unweighted validation cross-entropy.


In [ ]:
def run_training_epoch(model, loader, optimizer, max_batches=None):
    model.train()
    objective_numerator = 0.0
    total_correct = 0
    total_valid = 0

    progress = tqdm(loader, desc="Training", leave=False)
    for batch_index, batch in enumerate(progress):
        if max_batches is not None and batch_index >= max_batches:
            break

        inputs = batch["input_ids"].to(DEVICE, non_blocking=True)
        targets = batch["targets"].to(DEVICE, non_blocking=True)
        mask = batch["mask"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = masked_training_cross_entropy(logits, targets, mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        valid_count = int(mask.sum())
        predictions = logits.argmax(dim=-1)
        correct = int(((predictions == targets) & mask).sum())
        objective_numerator += float(loss.detach()) * valid_count
        total_correct += correct
        total_valid += valid_count
        progress.set_postfix(
            loss=f"{objective_numerator / max(1, total_valid):.4f}",
            acc=f"{total_correct / max(1, total_valid):.3f}",
        )

    return {
        "objective_loss": objective_numerator / max(1, total_valid),
        "accuracy": total_correct / max(1, total_valid),
        "valid_slots": total_valid,
    }

@torch.inference_mode()
def run_stateful_evaluation_epoch(model, dataset, max_windows=None):
    'Retain state within each song and compute unweighted cross-entropy and perplexity.'
    model.eval()
    cross_entropy_sum = 0.0
    total_correct = 0
    total_valid = 0
    state = None
    active_record_id = None
    expected_token_start = 0
    completed_record_ids = set()

    progress = tqdm(range(len(dataset)), desc="Stateful validation", leave=False)
    for dataset_index in progress:
        if max_windows is not None and dataset_index >= max_windows:
            break

        record_id = str(dataset.record_ids[dataset_index])
        token_start = int(dataset.token_starts[dataset_index])
        length = int(dataset.lengths[dataset_index])
        if record_id != active_record_id:
            if active_record_id is not None:
                completed_record_ids.add(active_record_id)
            if record_id in completed_record_ids:
                raise RuntimeError("Evaluation windows for a song are not contiguous")
            if token_start != 0:
                raise RuntimeError("First evaluation window does not start at 0")
            active_record_id = record_id
            expected_token_start = 0
            state = None
        if token_start != expected_token_start:
            raise RuntimeError(
                f"Non-contiguous windows for {record_id}: "
                f"expected {expected_token_start}, got {token_start}"
            )

        inputs = torch.as_tensor(
            dataset.input_ids[dataset_index, :length],
            dtype=torch.long,
            device=DEVICE,
        ).unsqueeze(0)
        targets = torch.as_tensor(
            dataset.targets[dataset_index, :length],
            dtype=torch.long,
            device=DEVICE,
        ).unsqueeze(0)
        mask = torch.as_tensor(
            dataset.masks[dataset_index, :length],
            dtype=torch.bool,
            device=DEVICE,
        ).unsqueeze(0)

        logits, state = model(inputs, state=state, return_state=True)
        loss_sum, valid_count = cross_entropy_sum_and_count(
            logits, targets, mask
        )
        predictions = logits.argmax(dim=-1)
        cross_entropy_sum += loss_sum
        total_correct += int(((predictions == targets) & mask).sum())
        total_valid += valid_count
        expected_token_start = token_start + length

        current_ce = cross_entropy_sum / max(1, total_valid)
        progress.set_postfix(
            ce=f"{current_ce:.4f}",
            ppl=f"{math.exp(min(current_ce, 50.0)):.2f}",
            acc=f"{total_correct / max(1, total_valid):.3f}",
        )

    cross_entropy = cross_entropy_sum / max(1, total_valid)
    return {
        "cross_entropy": cross_entropy,
        "perplexity": math.exp(min(cross_entropy, 50.0)),
        "accuracy": total_correct / max(1, total_valid),
        "valid_slots": total_valid,
    }

reset_random_state()


train_loader_generator.manual_seed(RANDOM_SEED)
model = make_model().to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1
)

checkpoint_path = EXPERIMENT_ROOT / "best_causal_model.pt"
history_rows = []
best_validation_cross_entropy = math.inf
patience_counter = 0
checkpoint_saved_this_run = False

experiment_config = {
    "active_top_n": ACTIVE_TOP_N,
    "random_seed": RANDOM_SEED,
    "test_split_sha256": TEST_SPLIT_SHA256,
    "split_assignment_sha256": SPLIT_ASSIGNMENT_SHA256,
    "input_interface_sha256": INPUT_INTERFACE_SHA256,
    "window_tokens": WINDOW_TOKENS,
    "train_hop_tokens": TRAIN_HOP_TOKENS,
    "eval_hop_tokens": EVAL_HOP_TOKENS,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "grad_clip": GRAD_CLIP,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "max_train_batches": MAX_TRAIN_BATCHES,
    "max_validation_batches": MAX_VALIDATION_BATCHES,
}

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()
    train_result = run_training_epoch(
        model, train_loader, optimizer, MAX_TRAIN_BATCHES
    )
    validation_result = run_stateful_evaluation_epoch(
        model, validation_dataset, MAX_VALIDATION_BATCHES
    )
    scheduler.step(validation_result["cross_entropy"])

    row = {
        "epoch": epoch,
        "train_objective_loss": train_result["objective_loss"],
        "train_accuracy": train_result["accuracy"],
        "validation_cross_entropy": validation_result["cross_entropy"],
        "validation_perplexity": validation_result["perplexity"],
        "validation_accuracy": validation_result["accuracy"],
        "learning_rate": optimizer.param_groups[0]["lr"],
        "minutes": (time.time() - epoch_start) / 60.0,
    }
    history_rows.append(row)
    print(
        f"Top {ACTIVE_TOP_N} | Epoch {epoch:02d} | "
        f"train loss {row['train_objective_loss']:.4f}, "
        f"acc {row['train_accuracy']:.3f} | "
        f"val CE {row['validation_cross_entropy']:.4f}, "
        f"PPL {row['validation_perplexity']:.2f}, "
        f"acc {row['validation_accuracy']:.3f} | "
        f"{row['minutes']:.1f} min"
    )

    if (
        validation_result["cross_entropy"]
        < best_validation_cross_entropy - 1e-4
    ):
        best_validation_cross_entropy = validation_result["cross_entropy"]
        patience_counter = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "validation_cross_entropy": best_validation_cross_entropy,
                "validation_perplexity": validation_result["perplexity"],
                "model_config": {
                    "input_vocab_size": MODEL_INPUT_VOCAB_SIZE,
                    "chord_vocab_size": CHORD_VOCAB_SIZE,
                    "pad_token_id": MIDITOK_PAD_ID,
                    "embedding_dim": EMBEDDING_DIM,
                    "hidden_size": HIDDEN_SIZE,
                    "num_layers": NUM_LAYERS,
                    "dropout": DROPOUT,
                    "bidirectional": False,
                },
                "experiment_config": experiment_config,
                "live_input_schema": LIVE_INPUT_SCHEMA,
                "window_schema": WINDOW_SCHEMA,
                "encoding_revision": encoding_revision,
            },
            checkpoint_path,
        )
        checkpoint_saved_this_run = True
        print("  saved new best causal checkpoint")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

history_df = pd.DataFrame(history_rows)
history_df.to_csv(EXPERIMENT_ROOT / "training_history.csv", index=False)
display(history_df)

In [ ]:
if not checkpoint_saved_this_run or not checkpoint_path.exists():
    raise RuntimeError("This run did not create a finite best checkpoint")

checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print("Loaded best epoch:", checkpoint["epoch"])
print(
    "Best validation cross-entropy:",
    checkpoint["validation_cross_entropy"],
)
print("Best validation perplexity:", checkpoint["validation_perplexity"])

figure, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(
    history_df["epoch"],
    history_df["train_objective_loss"],
    label="train objective",
)
axes[0].plot(
    history_df["epoch"],
    history_df["validation_cross_entropy"],
    label="validation CE",
)
axes[0].set_title("Cross-entropy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(
    history_df["epoch"],
    history_df["validation_perplexity"],
    color="tab:orange",
)
axes[1].set_title("Validation perplexity")
axes[1].set_xlabel("Epoch")

axes[2].plot(
    history_df["epoch"], history_df["train_accuracy"], label="train"
)
axes[2].plot(
    history_df["epoch"],
    history_df["validation_accuracy"],
    label="validation",
)
axes[2].set_title("24-PPQ slot accuracy")
axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0, 1)
axes[2].legend()

figure.suptitle(f"Top {ACTIVE_TOP_N} causal LSTM learning curves")
figure.tight_layout()
figure.savefig(
    REPORT_ROOT / "learning_curves.png", dpi=180, bbox_inches="tight"
)
plt.show()

## Final stateful causal test

Every supervised test position is evaluated once while LSTM state is retained across consecutive windows of a song. Metrics distinguish overall and retained-only accuracy, macro F1 including and excluding `<UNK>`, chord-onset accuracy, unweighted cross-entropy and perplexity.


In [ ]:
@torch.inference_mode()
def evaluate_test_statefully(model, dataset):
    model.eval()
    cross_entropy_sum = 0.0
    retained_cross_entropy_sum = 0.0
    valid_total = 0
    retained_total = 0
    top3_correct = 0
    retained_top3_correct = 0
    onset_correct = 0
    onset_total = 0
    retained_onset_correct = 0
    retained_onset_total = 0
    truth_parts = []
    prediction_parts = []
    state = None
    active_record_id = None
    expected_token_start = 0
    completed_record_ids = set()

    for dataset_index in tqdm(
        range(len(dataset)), desc=f"Top {ACTIVE_TOP_N} stateful test"
    ):
        record_id = str(dataset.record_ids[dataset_index])
        token_start = int(dataset.token_starts[dataset_index])
        length = int(dataset.lengths[dataset_index])
        if record_id != active_record_id:
            if active_record_id is not None:
                completed_record_ids.add(active_record_id)
            if record_id in completed_record_ids:
                raise RuntimeError("Test windows for a song are not contiguous")
            if token_start != 0:
                raise RuntimeError("First test window does not start at 0")
            active_record_id = record_id
            expected_token_start = 0
            state = None
        if token_start != expected_token_start:
            raise RuntimeError(
                f"Non-contiguous test windows for {record_id}: "
                f"expected {expected_token_start}, got {token_start}"
            )

        inputs = torch.as_tensor(
            dataset.input_ids[dataset_index, :length],
            dtype=torch.long,
            device=DEVICE,
        ).unsqueeze(0)
        targets = torch.as_tensor(
            dataset.targets[dataset_index, :length],
            dtype=torch.long,
            device=DEVICE,
        ).unsqueeze(0)
        mask = torch.as_tensor(
            dataset.masks[dataset_index, :length],
            dtype=torch.bool,
            device=DEVICE,
        ).unsqueeze(0)
        onset = torch.as_tensor(
            dataset.onsets[dataset_index, :length],
            dtype=torch.bool,
            device=DEVICE,
        ).unsqueeze(0)

        logits, state = model(inputs, state=state, return_state=True)
        predictions = logits.argmax(dim=-1)

        valid_logits = logits[mask]
        valid_targets = targets[mask]
        valid_predictions = predictions[mask]
        valid_count = int(valid_targets.numel())
        if valid_count:
            cross_entropy_sum += float(
                F.cross_entropy(
                    valid_logits, valid_targets, reduction="sum"
                )
            )
            valid_total += valid_count

        retained_mask = valid_targets != CHORD_UNK_ID
        retained_count = int(retained_mask.sum())
        if retained_count:
            retained_cross_entropy_sum += float(
                F.cross_entropy(
                    valid_logits[retained_mask],
                    valid_targets[retained_mask],
                    reduction="sum",
                )
            )
            retained_total += retained_count

        k = min(3, CHORD_VOCAB_SIZE)
        top3 = valid_logits.topk(k, dim=-1).indices
        top3_hits = (top3 == valid_targets.unsqueeze(-1)).any(dim=-1)
        top3_correct += int(top3_hits.sum())
        retained_top3_correct += int(top3_hits[retained_mask].sum())

        onset_mask = onset & mask
        retained_onset_mask = onset_mask & (targets != CHORD_UNK_ID)
        onset_correct += int(((predictions == targets) & onset_mask).sum())
        onset_total += int(onset_mask.sum())
        retained_onset_correct += int(
            ((predictions == targets) & retained_onset_mask).sum()
        )
        retained_onset_total += int(retained_onset_mask.sum())

        truth_parts.append(valid_targets.cpu().numpy())
        prediction_parts.append(valid_predictions.cpu().numpy())
        expected_token_start = token_start + length

    y_true = np.concatenate(truth_parts).astype(int)
    y_pred = np.concatenate(prediction_parts).astype(int)

    retained_truth_mask = y_true != CHORD_UNK_ID
    retained_y_true = y_true[retained_truth_mask]
    retained_y_pred = y_pred[retained_truth_mask]
    labels_including_unk = np.asarray(
        sorted(set(np.unique(y_true)) - {CHORD_PAD_ID}), dtype=int
    )
    labels_excluding_unk = np.asarray(
        sorted(set(np.unique(retained_y_true)) - {CHORD_PAD_ID}), dtype=int
    )

    macro_including_unk = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels_including_unk,
        average="macro",
        zero_division=0,
    )
    macro_excluding_unk = precision_recall_fscore_support(
        retained_y_true,
        retained_y_pred,
        labels=labels_excluding_unk,
        average="macro",
        zero_division=0,
    )
    weighted_including_unk = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels_including_unk,
        average="weighted",
        zero_division=0,
    )
    weighted_excluding_unk = precision_recall_fscore_support(
        retained_y_true,
        retained_y_pred,
        labels=labels_excluding_unk,
        average="weighted",
        zero_division=0,
    )
    unk_precision, unk_recall, unk_f1, _ = (
        precision_recall_fscore_support(
            y_true == CHORD_UNK_ID,
            y_pred == CHORD_UNK_ID,
            average="binary",
            zero_division=0,
        )
    )

    cross_entropy = cross_entropy_sum / max(1, valid_total)
    retained_cross_entropy = (
        retained_cross_entropy_sum / max(1, retained_total)
    )
    perplexity = math.exp(min(cross_entropy, 50.0))
    retained_perplexity = math.exp(
        min(retained_cross_entropy, 50.0)
    )
    
    uniform_cross_entropy = math.log(CHORD_VOCAB_SIZE)

    train_coverage = next(
        row
        for row in encoding_summary["coverage_by_split"]
        if row["split"] == "train"
    )
    test_coverage = next(
        row
        for row in encoding_summary["coverage_by_split"]
        if row["split"] == "test"
    )

    metrics = {
        "evaluation_mode": "stateful causal streaming by song",
        "excluding_unk_metrics_scope": "positions whose gold target is retained, not UNK",
        "active_top_n": ACTIVE_TOP_N,
        "chord_vocab_size_including_pad": CHORD_VOCAB_SIZE,
        "supervised_chord_classes_including_unk": (
            EXPECTED_SUPERVISED_CHORD_CLASSES
        ),
        "cross_entropy_loss": float(cross_entropy),
        "perplexity": float(perplexity),
        "retained_only_cross_entropy_loss": float(retained_cross_entropy),
        "retained_only_perplexity": float(retained_perplexity),
        "uniform_baseline_cross_entropy": float(uniform_cross_entropy),
        "uniform_baseline_perplexity": float(CHORD_VOCAB_SIZE),
        "uniform_baseline_output_class_count": int(CHORD_VOCAB_SIZE),
        "normalised_cross_entropy_vs_uniform": float(
            cross_entropy / uniform_cross_entropy
        ),
        "slot_accuracy": float((y_true == y_pred).mean()),
        "retained_only_slot_accuracy": float(
            (y_true[retained_truth_mask] == y_pred[retained_truth_mask]).mean()
        ),
        "top3_accuracy": top3_correct / max(1, valid_total),
        "retained_only_top3_accuracy": (
            retained_top3_correct / max(1, retained_total)
        ),
        "macro_precision_including_unk": float(macro_including_unk[0]),
        "macro_recall_including_unk": float(macro_including_unk[1]),
        "macro_f1_including_unk": float(macro_including_unk[2]),
        "macro_precision_excluding_unk": float(macro_excluding_unk[0]),
        "macro_recall_excluding_unk": float(macro_excluding_unk[1]),
        "macro_f1_excluding_unk": float(macro_excluding_unk[2]),
        "weighted_f1_including_unk": float(weighted_including_unk[2]),
        "weighted_f1_excluding_unk": float(weighted_excluding_unk[2]),
        
        "macro_f1": float(macro_excluding_unk[2]),
        "weighted_f1": float(weighted_excluding_unk[2]),
        "chord_onset_accuracy": onset_correct / max(1, onset_total),
        "chord_onset_accuracy_excluding_unk": (
            retained_onset_correct / max(1, retained_onset_total)
        ),
        "unk_target_rate": float((y_true == CHORD_UNK_ID).mean()),
        "unk_prediction_rate": float((y_pred == CHORD_UNK_ID).mean()),
        "unk_precision": float(unk_precision),
        "unk_recall": float(unk_recall),
        "unk_f1": float(unk_f1),
        "invalid_pad_prediction_rate": float(
            (y_pred == CHORD_PAD_ID).mean()
        ),
        "majority_baseline_slot_accuracy": float(
            (y_true == majority_chord_id).mean()
        ),
        "majority_chord_id": majority_chord_id,
        "majority_chord_label": majority_chord_label,
        "encoding_train_slot_coverage_pct": float(
            train_coverage["retained_slot_coverage_pct"]
        ),
        "encoding_test_unk_slot_pct": float(test_coverage["unk_slot_pct"]),
        "supervised_test_slots": int(valid_total),
        "retained_test_slots": int(retained_total),
        "test_chord_onsets": int(onset_total),
        "retained_test_chord_onsets": int(retained_onset_total),
        "best_epoch": int(checkpoint["epoch"]),
        "best_validation_cross_entropy": float(
            checkpoint["validation_cross_entropy"]
        ),
        "best_validation_perplexity": float(
            checkpoint["validation_perplexity"]
        ),
        "trainable_parameter_count": int(parameter_count),
        "experiment_config": experiment_config,
        "encoding_revision": encoding_revision,
    }
    return metrics, y_true, y_pred

test_metrics, y_true, y_pred = evaluate_test_statefully(
    model, test_dataset
)
with (EXPERIMENT_ROOT / "test_metrics.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(test_metrics, handle, indent=2)

display(pd.Series(test_metrics, name="value").to_frame())

score_names = [
    "slot_accuracy",
    "retained_only_slot_accuracy",
    "macro_f1_including_unk",
    "macro_f1_excluding_unk",
    "chord_onset_accuracy",
    "chord_onset_accuracy_excluding_unk",
]
score_values = [test_metrics[name] for name in score_names]
figure, axes = plt.subplots(1, 2, figsize=(15, 5))
bars = axes[0].bar(score_names, score_values)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Score")
axes[0].set_title(f"Top {ACTIVE_TOP_N}: predictive metrics")
axes[0].tick_params(axis="x", rotation=35)
for bar, value in zip(bars, score_values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        fontsize=8,
    )

uncertainty_names = [
    "cross_entropy_loss",
    "perplexity",
    "normalised_cross_entropy_vs_uniform",
]
uncertainty_values = [test_metrics[name] for name in uncertainty_names]
bars = axes[1].bar(uncertainty_names, uncertainty_values, color="tab:orange")
axes[1].set_title(f"Top {ACTIVE_TOP_N}: uncertainty metrics")
axes[1].tick_params(axis="x", rotation=30)
for bar, value in zip(bars, uncertainty_values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + max(uncertainty_values) * 0.02,
        f"{value:.3f}",
        ha="center",
        fontsize=8,
    )

figure.tight_layout()
figure.savefig(
    REPORT_ROOT / "test_metrics.png", dpi=180, bbox_inches="tight"
)
plt.show()

## Held-out causal prediction examples

Examples report gold and predicted unknown rates. `<UNK>` is an evaluation class rather than a playable chord and would require a product-level fallback policy.


In [ ]:
def collapse_consecutive(ids, limit=24):
    collapsed = []
    previous = None
    for token_id in ids:
        token_id = int(token_id)
        if token_id != previous:
            collapsed.append(ID_TO_CHORD.get(token_id, f"ID_{token_id}"))
            previous = token_id
        if len(collapsed) >= limit:
            break
    return " -> ".join(collapsed)

song_start_indices = np.flatnonzero(test_dataset.token_starts == 0)
if len(song_start_indices) == 0:
    raise RuntimeError("No test window begins at a song boundary")
selected_positions = np.linspace(
    0,
    len(song_start_indices) - 1,
    num=min(5, len(song_start_indices)),
    dtype=int,
)
candidate_indices = song_start_indices[selected_positions]

example_rows = []
model.eval()
with torch.no_grad():
    for dataset_index in candidate_indices:
        dataset_index = int(dataset_index)
        length = int(test_dataset.lengths[dataset_index])
        inputs = torch.as_tensor(
            test_dataset.input_ids[dataset_index, :length],
            dtype=torch.long,
            device=DEVICE,
        ).unsqueeze(0)
        logits = model(inputs)[0].cpu()
        predictions = logits.argmax(dim=-1).numpy()
        mask = test_dataset.masks[dataset_index, :length].astype(bool)
        onset = test_dataset.onsets[dataset_index, :length].astype(bool) & mask
        targets = test_dataset.targets[dataset_index, :length]

        grid_true = targets[mask]
        grid_pred = predictions[mask]
        gold_onsets = targets[onset]
        pred_onsets = predictions[onset]
        example_rows.append(
            {
                "record_id": test_dataset.record_ids[dataset_index],
                "window_token_start": int(test_dataset.token_starts[dataset_index]),
                "window_slot_accuracy": float((grid_true == grid_pred).mean()),
                "gold_unk_rate": float((grid_true == CHORD_UNK_ID).mean()),
                "predicted_unk_rate": float((grid_pred == CHORD_UNK_ID).mean()),
                "gold_at_chord_onsets": collapse_consecutive(gold_onsets),
                "predicted_at_gold_onsets": collapse_consecutive(pred_onsets),
                "free_predicted_progression": collapse_consecutive(grid_pred),
            }
        )

examples_df = pd.DataFrame(example_rows)
examples_df.to_csv(EXPERIMENT_ROOT / "test_examples.csv", index=False)
display(examples_df)

## Stateful streaming inference and model-only latency

Live input consists of pitch events at note-on, `NOTE_OFF` at release, and one clock token per 24-PPQ step. Hidden and cell states are carried between calls. The benchmark measures PyTorch model computation only and excludes browser, MIDI, network and audio-scheduling latency.


In [ ]:
def synchronise_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

model.eval()
best_causality_checks = run_causality_checks(
    model, first_batch["input_ids"].to(DEVICE)
)

demo_index = 0
demo_length = int(test_dataset.lengths[demo_index])
demo_tokens = test_dataset.input_ids[demo_index, :demo_length].astype(int)
state = None
stream_rows = []
with torch.inference_mode():
    for token_position, token_id in enumerate(demo_tokens[: min(demo_length, 512)]):
        chord_logits, state = model.stream_step(int(token_id), state=state)
        if int(token_id) == MIDITOK_CLOCK_ID:
            chord_id = int(chord_logits.argmax(dim=-1).item())
            stream_rows.append(
                {
                    "token_position": token_position,
                    "predicted_chord": ID_TO_CHORD.get(chord_id, f"ID_{chord_id}"),
                }
            )

display(pd.DataFrame(stream_rows[:12]))
print("Streaming chord outputs in demo window:", len(stream_rows))

benchmark_tokens = demo_tokens
if len(benchmark_tokens) == 0:
    raise RuntimeError("No tokens available for latency benchmark")

state = None
with torch.inference_mode():
    for index in range(LATENCY_WARMUP_STEPS):
        token_id = int(benchmark_tokens[index % len(benchmark_tokens)])
        _, state = model.stream_step(token_id, state=state)
    synchronise_device()

    state = None
    latency_ms = []
    for index in range(LATENCY_MEASURE_STEPS):
        token_id = int(benchmark_tokens[index % len(benchmark_tokens)])
        synchronise_device()
        start = time.perf_counter()
        chord_logits, state = model.stream_step(token_id, state=state)
        _ = int(chord_logits.argmax(dim=-1).item())
        synchronise_device()
        latency_ms.append((time.perf_counter() - start) * 1000.0)

latency_report = {
    "device": str(DEVICE),
    "device_name": (
        torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
    ),
    "measurement_count": int(len(latency_ms)),
    "median_model_step_ms": float(np.median(latency_ms)),
    "p95_model_step_ms": float(np.percentile(latency_ms, 95)),
    "mean_model_step_ms": float(np.mean(latency_ms)),
    "grid_tick_ms_at_120_bpm": float(60000.0 / (120 * SELECTED_PPQ)),
    "scope": "one-token PyTorch model compute; excludes browser/MIDI/network/audio",
}

causality_and_latency = {
    "architecture": "two-layer causal unidirectional LSTM",
    "live_input_schema": LIVE_INPUT_SCHEMA,
    "causality_checks": best_causality_checks,
    "latency": latency_report,
}
with (EXPERIMENT_ROOT / "causality_and_latency.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(causality_and_latency, handle, indent=2)

display(pd.Series(latency_report, name="value").to_frame())
print("Causality checks:", best_causality_checks)


## Generate the report for one Top-N experiment


In [ ]:
preprocessing_summary_path = Path(
    encoding_summary.get("preprocessing_summary_path", "")
)
if not preprocessing_summary_path.is_file():
    preprocessing_summary_path = (
        OUTPUT_ROOT / "preprocessing" / "xml_only_v1_summary.json"
    )
with preprocessing_summary_path.open(encoding="utf-8") as handle:
    preprocessing_summary = json.load(handle)

first_example = examples_df.iloc[0].to_dict()
top_three_progressions = musical_ngram_stats_df[
    musical_ngram_stats_df["n"] == 3
].head(3)
progression_text = "; ".join(
    f"{row.progression} ({int(row.count)})"
    for row in top_three_progressions.itertuples(index=False)
)

report_text = f"""# Top {ACTIVE_TOP_N} causal LSTM experiment

## 1. Encoding and vocabulary

- MXL files attempted: {preprocessing_summary['source_mxl_count']:,}
- Accepted and encoded songs: {encoding_summary['song_count']:,}
- Vocabulary selected from: training split only
- Retained Harte labels: {ACTIVE_TOP_N}
- Supervised chord classes: {EXPECTED_SUPERVISED_CHORD_CLASSES} ({ACTIVE_TOP_N} retained + one UNK)
- Technical output vocabulary including PAD: {CHORD_VOCAB_SIZE}
- Training retained-slot coverage: {test_metrics['encoding_train_slot_coverage_pct']:.4f}%
- Test UNK target rate: {100 * test_metrics['unk_target_rate']:.4f}%
- Gold chord leakage: 0; song-group leakage: 0

## 2. Real-time-safe input

- MIDITok TSD with a 24-PPQ alignment clock.
- Live events: Pitch at note-on, NOTE_OFF at release, and one TimeShift clock per grid step.
- Offline Duration tokens visible to model: 0.
- Most frequent retained chord trigrams: {progression_text}

## 3. Model and training

- Two-layer unidirectional LSTM (`bidirectional=False`).
- Embedding / hidden size: {EMBEDDING_DIM} / {HIDDEN_SIZE}; dropout: {DROPOUT}.
- Training windows: {WINDOW_TOKENS} tokens with {TRAIN_HOP_TOKENS}-token hop.
- Validation/test windows: non-overlapping {EVAL_HOP_TOKENS}-token hop with state carried by song.
- Best epoch: {int(checkpoint['epoch'])}.
- Best validation cross-entropy: {checkpoint['validation_cross_entropy']:.4f}.
- Best validation perplexity: {checkpoint['validation_perplexity']:.3f}.

## 4. Stateful causal test results

- Cross-Entropy Loss: {test_metrics['cross_entropy_loss']:.4f}
- Perplexity: {test_metrics['perplexity']:.3f}
- Normalised CE versus uniform baseline: {test_metrics['normalised_cross_entropy_vs_uniform']:.4f}
- Overall slot accuracy: {test_metrics['slot_accuracy']:.4f}
- Retained-only slot accuracy: {test_metrics['retained_only_slot_accuracy']:.4f}
- Macro F1 including UNK: {test_metrics['macro_f1_including_unk']:.4f}
- Macro F1 excluding UNK: {test_metrics['macro_f1_excluding_unk']:.4f}
- Chord-onset accuracy including UNK: {test_metrics['chord_onset_accuracy']:.4f}
- Chord-onset accuracy excluding UNK: {test_metrics['chord_onset_accuracy_excluding_unk']:.4f}
- UNK target / prediction rate: {100 * test_metrics['unk_target_rate']:.3f}% / {100 * test_metrics['unk_prediction_rate']:.3f}%
- Majority baseline slot accuracy: {test_metrics['majority_baseline_slot_accuracy']:.4f}

## 5. Causality and latency

- Future perturbation max earlier-output difference: {best_causality_checks['future_perturbation_max_abs_diff']:.8f}
- Streaming equivalence max difference: {best_causality_checks['streaming_equivalence_max_abs_diff']:.8f}
- Median / p95 model step latency: {latency_report['median_model_step_ms']:.3f} / {latency_report['p95_model_step_ms']:.3f} ms
- Device: {latency_report['device_name']}

## 6. Interpretation

- Macro F1 excluding UNK is computed only on retained-target positions and is the main class-balance metric; including-UNK F1 is also reported so the merged unknown class cannot silently inflate the result.
- Perplexity is reported as requested, but Top 200 and Top 250 have different class counts. It must be interpreted with normalised CE, retained-only metrics and coverage rather than used alone.
- UNK is an evaluation class, not a playable chord. A product fallback is still required.
"""

group_report_path = EXPERIMENT_ROOT / "GROUP_MEETING_UPDATE.md"
group_report_path.write_text(report_text, encoding="utf-8")
display(Markdown(report_text))
print("Saved report:", group_report_path)
print("Saved checkpoint:", checkpoint_path)
print("Saved test metrics:", EXPERIMENT_ROOT / "test_metrics.json")

## Method boundary and two-run protocol

The experiment establishes learnable signal, architectural causality, stateful equivalence and model-only latency; it does not establish subjective musical quality. Top-200 and Top-250 must be run under identical settings, with no training-batch limits.


## Automated Top-200 versus Top-250 comparison

The comparison prioritises macro F1 excluding `<UNK>`, chord-onset accuracy excluding `<UNK>`, retained-only slot accuracy and coverage. Cross-entropy and perplexity are reported as uncertainty evidence but are interpreted cautiously because the class counts differ.


In [ ]:
comparison_metric_paths = {
    top_n: (
        OUTPUT_ROOT
        / f"causal_lstm_tsd24_top{top_n}_unk_v3_audited"
        / "test_metrics.json"
    )
    for top_n in SUPPORTED_TOP_NS
}
missing_comparison_metrics = {
    top_n: str(path)
    for top_n, path in comparison_metric_paths.items()
    if not path.exists()
}

if missing_comparison_metrics:
    print('The Top-N comparison is incomplete; missing: ')
    for top_n, path in missing_comparison_metrics.items():
        print(f"- Top {top_n}: {path}")
    print('Complete the other full training run and rerun this cell.')
else:
    comparison_metrics = {}
    for top_n, path in comparison_metric_paths.items():
        comparison_metrics[top_n] = json.loads(
            path.read_text(encoding="utf-8")
        )

    config_keys = [
        "random_seed",
        "test_split_sha256",
        "split_assignment_sha256",
        "input_interface_sha256",
        "window_tokens",
        "train_hop_tokens",
        "eval_hop_tokens",
        "embedding_dim",
        "hidden_size",
        "num_layers",
        "dropout",
        "batch_size",
        "max_epochs",
        "early_stopping_patience",
        "learning_rate",
        "weight_decay",
        "grad_clip",
        "use_class_weights",
        "max_train_batches",
        "max_validation_batches",
    ]
    reference_config = comparison_metrics[200]["experiment_config"]
    config_mismatches = {}
    for top_n in SUPPORTED_TOP_NS:
        config = comparison_metrics[top_n]["experiment_config"]
        mismatches = {
            key: {
                "top200": reference_config.get(key),
                f"top{top_n}": config.get(key),
            }
            for key in config_keys
            if config.get(key) != reference_config.get(key)
        }
        if mismatches:
            config_mismatches[top_n] = mismatches
    if config_mismatches:
        raise RuntimeError(
            f"Top-N experiments are not controlled: {config_mismatches}"
        )

    comparison_columns = [
        "active_top_n",
        "chord_vocab_size_including_pad",
        "encoding_train_slot_coverage_pct",
        "encoding_test_unk_slot_pct",
        "unk_target_rate",
        "unk_prediction_rate",
        "cross_entropy_loss",
        "perplexity",
        "normalised_cross_entropy_vs_uniform",
        "slot_accuracy",
        "retained_only_slot_accuracy",
        "top3_accuracy",
        "macro_f1_including_unk",
        "macro_f1_excluding_unk",
        "weighted_f1_excluding_unk",
        "chord_onset_accuracy",
        "chord_onset_accuracy_excluding_unk",
        "majority_baseline_slot_accuracy",
        "best_epoch",
        "best_validation_cross_entropy",
        "trainable_parameter_count",
    ]
    comparison_rows = [
        {
            column: comparison_metrics[top_n][column]
            for column in comparison_columns
        }
        for top_n in SUPPORTED_TOP_NS
    ]
    comparison_df = pd.DataFrame(comparison_rows).set_index(
        "active_top_n"
    )
    display(comparison_df.T)

    delta_250_minus_200 = {
        column: float(comparison_df.loc[250, column])
        - float(comparison_df.loc[200, column])
        for column in comparison_df.columns
        if pd.api.types.is_numeric_dtype(comparison_df[column])
    }
    delta_df = pd.Series(
        delta_250_minus_200,
        name="Top250_minus_Top200",
    ).to_frame()
    display(delta_df)

    comparison_csv_path = COMPARISON_ROOT / "topn_model_comparison.csv"
    comparison_json_path = COMPARISON_ROOT / "topn_model_comparison.json"
    comparison_plot_path = COMPARISON_ROOT / "topn_model_comparison.png"
    comparison_df.reset_index().to_csv(comparison_csv_path, index=False)
    comparison_json_path.write_text(
        json.dumps(
            {
                "status": "CONTROLLED_COMPARISON_PASS",
                "top200": comparison_metrics[200],
                "top250": comparison_metrics[250],
                "delta_top250_minus_top200": delta_250_minus_200,
                "interpretation_warning": (
                    "Perplexity is vocabulary-size dependent; use macro F1 "
                    "excluding UNK, retained-only accuracy, onset accuracy "
                    "excluding UNK, normalised CE and coverage together."
                ),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    score_metrics = [
        "retained_only_slot_accuracy",
        "macro_f1_excluding_unk",
        "chord_onset_accuracy_excluding_unk",
    ]
    uncertainty_metrics = [
        "cross_entropy_loss",
        "normalised_cross_entropy_vs_uniform",
    ]
    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    comparison_df[score_metrics].T.plot(kind="bar", ax=axes[0])
    axes[0].set_ylim(0, 1)
    axes[0].set_title("Predictive metrics excluding UNK targets")
    axes[0].set_ylabel("Score")
    axes[0].legend(title="Top-N")
    axes[0].tick_params(axis="x", rotation=25)

    comparison_df[uncertainty_metrics].T.plot(kind="bar", ax=axes[1])
    axes[1].set_title("Uncertainty metrics")
    axes[1].legend(title="Top-N")
    axes[1].tick_params(axis="x", rotation=25)
    figure.tight_layout()
    figure.savefig(comparison_plot_path, dpi=180, bbox_inches="tight")
    plt.show()

    print("CONTROLLED TOP 200 / TOP 250 COMPARISON PASSED")
    print("Saved:", comparison_csv_path)
    print("Saved:", comparison_json_path)
    print("Saved:", comparison_plot_path)